# EEG preprocessing 
### Effect of prediction and attention on C1 component
Written by Maximilien Van Migem
and created on 02/09/2025 

In [1]:
# Import some libraries
import os
import numpy as np
import mne
import pandas as pd

%matplotlib qt 

# Subject selection and loading data
sub = 35
# reference = 'average'
reference = ['M1','M2'] # Mastoids

data_directory = 'C:/Users/mvmigem/Documents/data/project_2/'
rejected_channels_path = data_directory + 'rejected_channels.npy'
rejected_ica_path = data_directory + 'mastoid_rejected_ica.npy'
# rejected_ica_path = data_directory + 'average_rejected_ica.npy'

rejected_ica = np.load(rejected_ica_path, allow_pickle=True)[..., np.newaxis][0]
rejected_channels = np.load(rejected_channels_path, allow_pickle=True)[..., np.newaxis][0]

current_file_path = data_directory + f'raw/sub-{sub:02}/eeg/main_{sub:02}.bdf'
current_behav_path = data_directory + f'raw/sub-{sub:02}/behav/participant_{sub}.csv'
behav_data = pd.read_csv(current_behav_path)

raw = mne.io.read_raw_bdf(current_file_path, preload = True)

# we rename the channels to their respective external electrodes
fix_chans = {'EXG1':'eye_above','EXG2':'eye_below',
             'EXG3':'eye_left','EXG4':'eye_right',
             'EXG5':'M1','EXG6':'M2'}
raw.rename_channels(fix_chans)

# we still have two exg channels which weren't actually recorded though (EXG7
# and EXG8) these are empty, so we'll drop them
raw.drop_channels(['EXG7', 'EXG8'])
print(raw.info['ch_names'])

# we'll also reset the channel types, so MNE knows what is 'brain' data
raw.set_channel_types({'M1':'eeg', 'M2':'eeg',
                       'eye_above':'eog', 'eye_below':'eog',
                       'eye_left':'eog', 'eye_right': 'eog'})

print(raw.info)

# # rereference to mastoids
# raw.set_eeg_reference(ref_channels = ['M1','M2'])

# rereference
raw.set_eeg_reference(ref_channels = reference)

raw.drop_channels(['M1','M2'])

## I'll drop the mastoids because I'm not using them
# raw.drop_channels(['mastoid_left', 'mastoid_right'])
# Set montage
montage = mne.channels.make_standard_montage('biosemi64')

# This dict is to rename the channel names to fit the montage
mon_chnames = montage.ch_names
raw_chnames = raw.info['ch_names']
rename_channels = dict(zip(raw_chnames[:64],mon_chnames))
raw.rename_channels(rename_channels)
# Set montage
raw.set_montage(montage)
# take another look
# raw.plot(n_channels = 64)

# let's take a look at what has been detected

# Downsampling variables (logic -> https://mne.tools/stable/auto_tutorials/preprocessing/30_filtering_resampling.html#best-practices)
current_sfreq = raw.info['sfreq']
desired_sfreq = 256  # Hz
decim = np.round(current_sfreq / desired_sfreq).astype(int)
obtained_sfreq = current_sfreq / decim
lowpass_freq = obtained_sfreq / 3.

raw_filtered = raw.copy().notch_filter(freqs = 50, fir_design = 'firwin', verbose=None, )
raw_filtered = raw_filtered.copy().filter(l_freq=0.1, h_freq=lowpass_freq) 

# Select event dict for condition
if behav_data['start_position'].isin([0, 2]).any():
    event_id = {
    'start_trial':99, 'pos1/seq':11, 'pos1/seq3':13, 
    'pos2/seq2':22, 'pos2/seq4':24,
    'pos3/seq1':31, 'pos3/seq3':33,
    'pos4/seq2':42, 'pos4/seq4':44,
    }
elif behav_data['start_position'].isin([1, 3]).any():
    # Event dict
    event_id = {
        'start_trial':99, 'pos1/seq2':12, 'pos1/seq4':14, 
        'pos2/seq1':21, 'pos2/seq3':23,
        'pos3/seq2':32, 'pos3/seq4':34,
        'pos4/seq1':41, 'pos4/seq3':43,
    }
events = mne.find_events(raw_filtered,shortest_event=1)

# we can visualise the paradigm (timecourse of the events), to confirm nothing
# weird has happened
fig = mne.viz.plot_events(events, 
                          sfreq = raw_filtered.info['sfreq'],
                          event_id = event_id)

# Define your threshold in seconds
threshold_ms = 1000
sfreq = raw_filtered.info['sfreq']  # Sampling frequency of your data
threshold_samples = int(threshold_ms / 1000 * sfreq)

# Calculate differences between consecutive events
event_times = events[:, 0]  # Extract the sample index (first column) of each event
time_diffs = np.diff(event_times)

# Identify where time differences exceed the threshold
long_gaps = time_diffs > threshold_samples

# Find the periods where the distance exceeds the threshold
indices_exceeding_threshold = np.where(long_gaps)[0]


# Create a list to hold the annotations
annotations = []

for idx in indices_exceeding_threshold:
    start_sample = events[idx, 0] + (0.52*sfreq) # Start of the period + 500ms for preceding trial
    end_sample = events[idx + 1, 0] - 1 # End of the period (-1 to avoid removing trial trigger)
    start_time = (start_sample / sfreq)  # Convert sample index to time in seconds and add padding for epoch
    duration = (end_sample - start_sample) / sfreq  # Duration in seconds

    # Create an annotation
    annotation = mne.Annotations(onset=start_time,
                                 duration=duration,
                                 description=f'bad_calibration_gap')
    
    # Append annotation to the list
    annotations.append(annotation)

# Convert the list of annotations to a single mne.Annotations object
if annotations:
    combined_annotations = annotations[0]
    for annotation in annotations[1:]:
        combined_annotations += annotation
    
    # Add the annotations to the raw object
    raw_filtered.set_annotations(combined_annotations)
else:
    print("No gaps exceeding the threshold were found.")

# Plotting for potential channel rejection
raw_filtered.plot_psd()
raw_filtered.plot(n_channels=64,block=True)

# Load the rejected channel
rejected_channels = np.load(rejected_channels_path, allow_pickle=True)[..., np.newaxis][0]
rejected_channels[f'subject_{sub}'] = raw_filtered.info['bads']  
np.save(rejected_channels_path, rejected_channels)
# Interpolate rejected channels
interp_filt_raw = raw_filtered.copy().interpolate_bads(reset_bads = False)

# ICA
ica = mne.preprocessing.ICA(n_components = 0.99)
ica.fit(interp_filt_raw,decim=2, verbose='error', reject_by_annotation=True)
ica.plot_components()

interp_filt_raw.plot(events=events,n_channels=64,)

Extracting EDF parameters from C:\Users\mvmigem\Documents\data\project_2\raw\sub-35\eeg\main_35.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1452031  =      0.000 ...  2835.998 secs...
['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 'eye_above', 'eye_below', 'eye_left', 'eye_right', 'M1', 'M2', 'Status']
<Info | 8 non-empty values
 bads: []
 ch_names: A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11, A12, A13, A14, ...
 chs: 66 EEG, 4 EOG, 1 Stimulus
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 104.0 Hz
 meas_date: 2025-07-15 16:42:31 UTC
 nchan:

C:\Users\mvmigem\AppData\Local\Temp\ipykernel_14516\1394764592.py:101: RuntimeWarning: event 65536 missing from event_id will be ignored
  fig = mne.viz.plot_events(events,
C:\Users\mvmigem\AppData\Local\Temp\ipykernel_14516\1394764592.py:101: RuntimeWarning: event 65635 missing from event_id will be ignored
  fig = mne.viz.plot_events(events,
C:\Users\mvmigem\AppData\Local\Temp\ipykernel_14516\1394764592.py:101: RuntimeWarning: event 65789 missing from event_id will be ignored
  fig = mne.viz.plot_events(events,
C:\Users\mvmigem\AppData\Local\Temp\ipykernel_14516\1394764592.py:101: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  fig = mne.viz.plot_events(events,


NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Setting 22266 of 1452032 (1.53%) samples to NaN, retaining 1429766 (98.47%) samples.
Effective window size : 4.000 (s)
At least one good data span is shorter than n_per_seg, and will be analyzed with a shorter window than the rest of the file.


c:\Users\mvmigem\AppData\Local\miniconda3\envs\mne\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 2048 is greater than input length  = 267, using nperseg = 267
  return _func(*args, **kwargs)
c:\Users\mvmigem\AppData\Local\miniconda3\envs\mne\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 2048 is greater than input length  = 1530, using nperseg = 1530
  return _func(*args, **kwargs)
c:\Users\mvmigem\AppData\Local\miniconda3\envs\mne\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 2048 is greater than input length  = 1563, using nperseg = 1563
  return _func(*args, **kwargs)
c:\Users\mvmigem\AppData\Local\miniconda3\envs\mne\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 2048 is greater than input length  = 607, using nperseg = 607
  return _func(*args, **kwargs)
c:\Users\mvmigem\AppData\Local\miniconda3\envs\mne\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 2048 

Plotting power spectral density (dB=True).
Using qt as 2D backend.
Channels marked as bad:
[np.str_('C6')]
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors


In [2]:
# Save the rejected ica's
exclude_ica = [0,3]

rejected_ica[f'subject_{sub}'] = exclude_ica
np.save(rejected_ica_path, rejected_ica)

# Exclude ica
ica.exclude=exclude_ica
ica.apply(interp_filt_raw)

# Sav the raw data
interp_filt_raw.save(f"C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-raw/clean-mastoid-{sub:02}-raw.fif", overwrite=True)

# Epoch data around stim onset
epochs_stimlock = mne.Epochs(interp_filt_raw, events, event_id = event_id,
    tmin = -0.1, tmax = 0.45, proj = False, baseline = (None,0), decim=decim, #from previous cell
    detrend = None, verbose = True, reject_by_annotation= True, preload = True)

# Save processed epoch file
# epochs_stimlock.save(f"C:/Users/mvmigem/Documents/data/project_1/preprocessed/average_ref/unpaired/main_clean_averageref_{sub:02}-epo.fif", overwrite=True)
epochs_stimlock.save(f"C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-epo/unpaired/clean-mastoid-{sub:02}-epo.fif", overwrite=True)

# Generate mne report to document preprocessing steps
report = mne.Report(title = f"prepocessing-sub-{sub:02}")
report.add_events(events=events, title='Events', sfreq = current_sfreq)
report.add_raw(raw=interp_filt_raw, title='Filtered and annotated')
report.add_ica(ica=ica,title='ICA', inst= interp_filt_raw, picks=exclude_ica)
report_path = data_directory + f'/preprocessed/mastoid-report/report-sub-{sub:02}.html'
report.save(report_path, overwrite=True)


Applying ICA to Raw instance
    Transforming to ICA space (53 components)
    Zeroing out 2 ICA components
    Projecting back using 63 PCA components
Writing C:\Users\mvmigem\Documents\data\project_2\preprocessed\mastoid-raw\clean-mastoid-35-raw.fif
Closing C:\Users\mvmigem\Documents\data\project_2\preprocessed\mastoid-raw\clean-mastoid-35-raw.fif
[done]
Not setting metadata
4457 matching events found
Setting baseline interval to [-0.09765625, 0.0] s
Applying baseline correction (mode: mean)
Using data from preloaded Raw for 4457 events and 282 original time points (prior to decimation) ...
25 bad epochs dropped
Embedding : jquery-3.6.0.min.js
Embedding : bootstrap.bundle.min.js
Embedding : bootstrap.min.css
Embedding : bootstrap-table/bootstrap-table.min.js
Embedding : bootstrap-table/bootstrap-table.min.css
Embedding : bootstrap-table/bootstrap-table-copy-rows.min.js
Embedding : bootstrap-table/bootstrap-table-export.min.js
Embedding : bootstrap-table/tableExport.min.js
Embedding :

C:\Users\mvmigem\AppData\Local\Temp\ipykernel_14516\2531507107.py:25: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  report.add_events(events=events, title='Events', sfreq = current_sfreq)


Using matplotlib as 2D backend.
Using qt as 2D backend.
Channels marked as bad:
[np.str_('C6')]
Applying ICA to Raw instance
    Transforming to ICA space (53 components)
    Zeroing out 2 ICA components
    Projecting back using 63 PCA components
    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
1375 matching events found
No baseline correction applied
0 projection items activated
    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
1375 matching events found
No baseline correction applied
0 projection items activated
Saving report to : C:\Users\mvmigem\Documents\data\project_2\preprocessed\mastoid-report\report-sub-35.html


'C:\\Users\\mvmigem\\Documents\\data\\project_2\\preprocessed\\mastoid-report\\report-sub-35.html'

In [ ]:
report_path = data_directory + f'/preprocessed/mastoid-report/report-sub-{sub:02}.html'
report.save(report_path, overwrite=True)

In [ ]:
# Epoch data around stim onset
epochs_stimlock = mne.Epochs(interp_filt_raw, events, event_id = event_id,
    tmin = -0.1, tmax = 0.45, proj = False, baseline = (None,0), decim=decim, #from previous cell
    detrend = None, verbose = True, reject_by_annotation= True, preload = True)

In [ ]:
raw_filtered.plot(n_channels=64,block=True)

In [ ]:
rejected_channels[f'subject_{sub}'] = interp_filt_raw.info['bads']
np.save(rejected_channels_path, rejected_channels)
interp_filt_raw = interp_filt_raw.copy().interpolate_bads(reset_bads = True)

In [ ]:
interp_filt_raw.plot(events=events,n_channels=64,)
ica.plot_components()

In [ ]:
report_path = data_directory + f'/preprocessed/mastoid-report/report-sub-{sub:02}.html'
report.save(report_path, overwrite=True)

In [ ]:
# Sav the raw data
clean_raw_path = f"C:/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-raw/clean-mastoid-{sub:02}-raw.fif"
interp_filt_raw = mne.io.read_raw_fif(clean_raw_path,preload=True)

In [ ]:
# ICA
ica = mne.preprocessing.ICA(n_components = 0.99)
ica.fit(interp_filt_raw,decim=2, verbose='error', reject_by_annotation=True)
ica.plot_components()

interp_filt_raw.plot(events=events,n_channels=64,)
